# **MACHINE LEARNING PROJECT**

Importing the necessary headers

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


Loading the datasets

1.caract
2.lieux
3.usagers
4.vehicules

In [2]:
# loading the datasets from the repository root
# (the previous paths pointed at Colab's /content and always used ';')
from pathlib import Path

def load_baac(filename):
    candidates = [Path(filename), Path("/content") / filename]
    path = next(p for p in candidates if p.exists())
    sample = path.read_bytes()[:500]
    sep = ";" if sample.count(b";") >= sample.count(b",") else ","
    return pd.read_csv(path, delimiter=sep)

c_df = load_baac("caract-2023.csv")
l_df = load_baac("lieux-2023.csv")
u_df = load_baac("usagers-2023.csv")
v_df = load_baac("vehicules-2023.csv")


In [3]:
# Num_Acc is the only key that ties the four BAAC tables together.
# The committed CSVs store a single collapsed id on every row, so a merge
# would be a cartesian product rather than the accident-level table.
id_counts = {
    "caract": c_df["Num_Acc"].nunique(dropna=False),
    "lieux": l_df["Num_Acc"].nunique(dropna=False),
    "vehicules": v_df["Num_Acc"].nunique(dropna=False),
    "usagers": u_df["Num_Acc"].nunique(dropna=False),
}
print("Num_Acc unique counts:", id_counts)
print("Num_Acc example:", c_df["Num_Acc"].iloc[0])
bad = [name for name, n in id_counts.items() if n < 2]
if bad:
    raise RuntimeError(
        "Cannot merge the BAAC tables: Num_Acc is not a usable accident id in "
        f"{bad} (example value {c_df['Num_Acc'].iloc[0]!r}, "
        f"dtype {c_df['Num_Acc'].dtype}). Every committed row carries the same "
        "collapsed id, so an inner join on Num_Acc cannot rebuild the analysis "
        "table. No model was fit and no metric was computed."
    )


Num_Acc unique counts: {'caract': 1, 'lieux': 1, 'vehicules': 1, 'usagers': 1}
Num_Acc example: 202300000000.0


RuntimeError: Cannot merge the BAAC tables: Num_Acc is not a usable accident id in ['caract', 'lieux', 'vehicules', 'usagers'] (example value np.float64(202300000000.0), dtype float64). Every committed row carries the same collapsed id, so an inner join on Num_Acc cannot rebuild the analysis table. No model was fit and no metric was computed.

Displaying the shape of the datasets

In [ ]:
print("Characteristics Dataset Shape:", c_df.shape)
print("lieux Dataset Shape:", l_df.shape)
print("usagers Dataset Shape:", u_df.shape)
print("vehicules Dataset Shape:", v_df.shape)

Accident characteristics -  dataset

In [ ]:
#Displaying the accident characteristics
print("Characteristics Data Info:")
print(c_df.info())
print(c_df.head())

Accident locations - dataset

In [ ]:
#Displaying the accident locations
print("Location Data Info:")
print(l_df.info())
print(l_df.head())

Users - dataset

In [ ]:

#Displaying the users
print("USERS Data Info:")
print(u_df.info())
print(u_df.head())

Vehicles - dataset

In [ ]:
#Displaying Vehicles Data
print("Vehicles Data Info:")
print(v_df.info())
print(v_df.head())

**MERGING THE DATASETS ( characteristics, location, users , vehicles)**

In [ ]:
# merging the datafiles
#1 merging c_df & l_df
merged_df= c_df.merge(l_df, on='Num_Acc', how= 'inner')
#2 merging merged_df & v_df
merged_df= merged_df.merge(v_df, on='Num_Acc', how='inner')
#3 merging merged_df & u_df
merged_df=merged_df.merge(u_df, on='Num_Acc', how='inner')

Displaying the shape of the merged dataset

In [ ]:
merged_df.shape

Displaying the merged_df

In [ ]:
print(merged_df.info())
print(merged_df.head())


Displaying the columns of the merged_df

In [ ]:
merged_df.columns

Displaying few rows of merged_df dataset

In [ ]:
merged_df.head(10)

**Clean the datastes**

In [ ]:
# calculating the missing percentage of missing data for each column
missing_percentage = (merged_df.isna().sum() / len(merged_df)) * 100
print("missing_percentage")
print(missing_percentage)

Visualize missing data

Dropping the columns with more than 50% missing data

In [ ]:
threshold = 50
columns_to_keep = missing_percentage[missing_percentage <= threshold].index
merged_df = merged_df[columns_to_keep]


In [ ]:
missing_percentage = (merged_df.isna().sum() / len(merged_df)) * 100
print("missing_percentage")
print(missing_percentage)

Fill remaining  missing values

In [ ]:
cleaned_df = merged_df[columns_to_keep]

for col in cleaned_df.columns:
    if cleaned_df[col].dtype in ['float64', 'int64']:
        cleaned_df[col].fillna(cleaned_df[col].median(),inplace=True)
    else:
        cleaned_df[col].fillna(cleaned_df[col].mode()[0],inplace=True)


In [ ]:
print(cleaned_df.info())

In [ ]:
#verify that all missing values are handled
print("Missing data after claeaning ")
print(cleaned_df.isna().sum())

ANALYZE DATA BY STATISTICS

In [ ]:
print("Descriptive Statistics:")
print(cleaned_df.describe())

Creation of "mortality" varaible  

In [ ]:
print("Creating mortality variable " )
cleaned_df['mortality'] = cleaned_df['grav'].apply(lambda x : 1 if x == 2 else 0)
print("Mortality Distribution")
print(cleaned_df['mortality'].value_counts())

Visualize the severity of accidents

In [ ]:
#Mortality Distribution
sns.countplot(x = 'mortality', data = cleaned_df)
plt.title('Mortality Distribution')
plt.show()

In [ ]:
sns.countplot(x='sexe', hue = 'mortality', data= cleaned_df)
plt.title('Mortality by Gender')
plt.show()

In [ ]:
sns.countplot(x='mois', hue = 'mortality', data= cleaned_df)
plt.title('Mortality by Month')
plt.show()

Additonal statistics for mortality

In [ ]:
print("\nAdditional Statistics for Mortality:")
print(cleaned_df.groupby('mortality').agg({'grav': ['count'], 'lum': ['mean'], 'surf': ['mean']}))

Final dataset info

In [ ]:
print("Final cleaned datset info:")
print(cleaned_df.info())
print("Final Dataset shape:", cleaned_df.shape)

# **Visualization and modeling**

**Visualize & distribution of mortality**

Age

In [ ]:
cleaned_df['age'] = 2023 - cleaned_df['an_nais']
sns.histplot(data=cleaned_df, x='age', hue='mortality', multiple='stack', bins=20, kde=False)
plt.title('Mortality by Age')
plt.show()


Vehicle type

In [ ]:
sns.countplot(x='catv', hue='mortality', data=cleaned_df)
plt.title('Mortality by Vehicle Type')
plt.xticks(rotation=45)
plt.show()


Analyze based on time

year

In [ ]:
# Accidents by year
sns.countplot(x='an', data=cleaned_df, order=sorted(cleaned_df['an'].unique()))
plt.title('Number of Accidents by Year')
plt.xlabel('Year')
plt.ylabel('Number of Accidents')
plt.xticks(rotation=45)
plt.show()


In [ ]:
# Group the dataset by 'an' and count the number of accidents for each year
accidents_by_year = cleaned_df.groupby('an').size()

# Print the number of accidents by year
print("Number of Accidents by Year:")
print(accidents_by_year)


month

In [ ]:
# Accidents by month
sns.countplot(x='mois', data=cleaned_df, order=range(1, 13))
plt.title('Number of Accidents by Month')
plt.xlabel('Month')
plt.ylabel('Number of Accidents')
plt.xticks(ticks=range(12), labels=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
plt.show()


In [ ]:
# Group by year and month
accidents_by_year_month = cleaned_df.groupby(['an', 'mois']).size().reset_index(name='count')

# Plot accidents by year and month
plt.figure(figsize=(12, 6))
sns.lineplot(x='mois', y='count', hue='an', data=accidents_by_year_month, marker='o')
plt.title('Number of Accidents by Month Across Years')
plt.xlabel('Month')
plt.ylabel('Number of Accidents')
plt.xticks(ticks=range(1, 13), labels=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
plt.legend(title='Year', loc='upper right')
plt.show()


TO find the most dangerous time of the day

In [ ]:

# Ensure 'hrmn' column is treated as a string
cleaned_df['hrmn'] = cleaned_df['hrmn'].astype(str)

# Add leading zeros to ensure uniform formatting (e.g., '600' becomes '06:00')
cleaned_df['hrmn'] = cleaned_df['hrmn'].str.zfill(4)

# Extract the hour part
cleaned_df['hour'] = cleaned_df['hrmn'].str[:2].astype(int)

# Group by hour and calculate the total accidents
hourly_analysis = cleaned_df.groupby('hour').agg(
    total_accidents=('mortality', 'count')  # Count all rows (accidents) per hour
).reset_index()

# Visualize total accidents
plt.figure(figsize=(12, 6))
sns.barplot(x='hour', y='total_accidents', data=hourly_analysis, color='blue', alpha=0.6)
plt.title('Total Accidents by Hour of the Day')
plt.xlabel('Hour of the Day')
plt.ylabel('Number of Accidents')
plt.show()

# Identify the most dangerous hour based on total accidents
most_dangerous_hour = hourly_analysis.loc[hourly_analysis['total_accidents'].idxmax()]
formatted_hour = f"{int(most_dangerous_hour['hour']):02}:00"

# Display the result
print("The most dangerous hour of the day:")
print(f"Hour: {formatted_hour}")
print(f"Total Accidents: {int(most_dangerous_hour['total_accidents'])}")


## Modeling

`mortality` is defined as `grav == 2`, so `grav` is the label. It is dropped before the split, and so is any column whose name contains `grav`. `an_nais` and `hrmn` are also dropped because `age` and `hour` already replace them.

The train/test split is done on the original rows. Models that rebalance use an `imblearn` pipeline: `StandardScaler`, then `SMOTE`, then the classifier. The sampler runs only inside `fit`, so synthetic rows stay in the training fold. The held-out test set is never resampled. Cross-validation calls the same pipeline, so each validation fold stays at the original class balance.


In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

leak_cols = [c for c in cleaned_df.columns if "grav" in str(c).lower()]
drop_cols = ["Num_Acc", "mortality", "an_nais", "hrmn"] + leak_cols
missing = [c for c in ["Num_Acc", "mortality", "grav"] if c not in cleaned_df.columns]
if missing:
    raise KeyError(f"expected columns are missing: {missing}")

X = cleaned_df.drop(columns=drop_cols)
y = cleaned_df["mortality"].astype(int)

still_leaking = [c for c in X.columns if "grav" in str(c).lower() or c == "mortality"]
if still_leaking:
    raise RuntimeError(f"leaking columns still in the feature matrix: {still_leaking}")
print("Dropped columns:", drop_cols)
print("grav in features:", "grav" in X.columns)
print("Feature count:", X.shape[1])

categorical_columns = X.select_dtypes(include=["object", "category"]).columns
for col in categorical_columns:
    X[col] = LabelEncoder().fit_transform(X[col].astype(str))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train rows: {len(y_train)} (fatal {int(y_train.sum())})")
print(f"Test rows: {len(y_test)} (fatal {int(y_test.sum())})")
print("SMOTE has not been fit. The test fold is the original class mix.")


def score_predictions(name, y_true, y_pred, y_prob):
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average=None, labels=[0, 1], zero_division=0
    )
    metrics = {
        "model": name,
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "fatal_precision": float(precision[1]),
        "fatal_recall": float(recall[1]),
        "fatal_f1": float(f1[1]),
        "pr_auc": float(average_precision_score(y_true, y_prob)),
    }
    print(f"\n{name}")
    print(
        f"accuracy {metrics['accuracy']:.4f} | "
        f"fatal precision {metrics['fatal_precision']:.4f} | "
        f"fatal recall {metrics['fatal_recall']:.4f} | "
        f"fatal f1 {metrics['fatal_f1']:.4f} | "
        f"PR-AUC {metrics['pr_auc']:.4f}"
    )
    print("Confusion matrix [rows true 0/1, cols pred 0/1]:")
    print(confusion_matrix(y_true, y_pred, labels=[0, 1]))
    print(classification_report(y_true, y_pred, digits=4, zero_division=0))
    return metrics


results = []


In [ ]:
# Original-split baselines. No SMOTE. grav is not a feature.
baseline_models = {
    "Logistic regression (original split)": LogisticRegression(max_iter=300, random_state=42),
    "Random forest default (original split)": RandomForestClassifier(random_state=42),
}

for name, model in baseline_models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]
    results.append(score_predictions(name, y_test, pred, prob))


In [ ]:
def make_smote_pipeline(clf):
    """Scale and resample inside fit only. predict() sees the real test rows."""
    return ImbPipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("smote", SMOTE(random_state=42)),
            ("clf", clf),
        ]
    )

smote_models = {
    "Logistic regression (SMOTE on train only)": LogisticRegression(
        class_weight="balanced", max_iter=300, random_state=42
    ),
    "Random forest shallow (SMOTE on train only)": RandomForestClassifier(
        n_estimators=50,
        max_depth=5,
        min_samples_split=150,
        min_samples_leaf=75,
        class_weight="balanced",
        random_state=42,
    ),
    "MLP 50-25 (SMOTE on train only)": MLPClassifier(
        hidden_layer_sizes=(50, 25),
        max_iter=300,
        alpha=0.01,
        early_stopping=True,
        random_state=42,
    ),
}

fitted = {}
for name, clf in smote_models.items():
    pipe = make_smote_pipeline(clf)
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    prob = pipe.predict_proba(X_test)[:, 1]
    results.append(score_predictions(name, y_test, pred, prob))
    fitted[name] = (pipe, prob)

# Decision threshold 0.6, scored on the original test fold (not a SMOTE test fold).
rf_name = "Random forest shallow (SMOTE on train only)"
rf_prob = fitted[rf_name][1]
rf_pred_06 = (rf_prob >= 0.6).astype(int)
results.append(
    score_predictions(
        "Random forest shallow, threshold 0.6 (original test)",
        y_test,
        rf_pred_06,
        rf_prob,
    )
)


In [ ]:
# 5-fold macro F1. The pipeline refits SMOTE on each training fold only.
cv_rows = []
for name, clf in smote_models.items():
    pipe = make_smote_pipeline(clf)
    scores = cross_val_score(
        pipe,
        X_train,
        y_train,
        cv=5,
        scoring="f1_macro",
        n_jobs=-1,
    )
    print(f"\n{name} CV macro F1: {scores}")
    print(f"mean {scores.mean():.4f}  std {scores.std():.4f}")
    cv_rows.append({"model": name, "cv_macro_f1_mean": float(scores.mean()), "cv_macro_f1_std": float(scores.std())})


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plot_rows = [row for row in results if row["model"] in smote_models]
labels = [row["model"].split(" (")[0] for row in plot_rows]
x = np.arange(len(labels))

plt.figure(figsize=(8, 5))
plt.bar(x, [row["accuracy"] for row in plot_rows], color="steelblue")
plt.xticks(x, labels, rotation=15, ha="right")
plt.ylim(0, 1)
plt.ylabel("Test accuracy")
plt.title("Test accuracy on the original held-out fold")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
width = 0.25
plt.bar(x - width, [row["fatal_precision"] for row in plot_rows], width, label="Fatal precision")
plt.bar(x, [row["fatal_recall"] for row in plot_rows], width, label="Fatal recall")
plt.bar(x + width, [row["pr_auc"] for row in plot_rows], width, label="PR-AUC")
plt.xticks(x, labels, rotation=15, ha="right")
plt.ylim(0, 1)
plt.ylabel("Score")
plt.title("Fatal-class precision, recall, and PR-AUC")
plt.legend()
plt.tight_layout()
plt.show()

print("Held-out test metrics")
for row in results:
    print(
        f"{row['model']}: accuracy {row['accuracy']:.4f}, "
        f"fatal precision {row['fatal_precision']:.4f}, "
        f"fatal recall {row['fatal_recall']:.4f}, "
        f"fatal f1 {row['fatal_f1']:.4f}, "
        f"PR-AUC {row['pr_auc']:.4f}"
    )


In [ ]:
# Same random-forest grid as the previous notebook, but SMOTE lives inside
# the pipeline and the search is fit on the training split only.
# 3 x 3 x 2 x 2 x 5 = 180 fits. The previous run was interrupted before it
# finished; this cell is the corrected search and has no saved score unless
# the fit below completes.
rf_search = ImbPipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("smote", SMOTE(random_state=42)),
        ("clf", RandomForestClassifier(class_weight="balanced", random_state=42)),
    ]
)
param_grid = {
    "clf__n_estimators": [50, 100, 200],
    "clf__max_depth": [10, 20, 30],
    "clf__min_samples_split": [2, 10],
    "clf__min_samples_leaf": [1, 5],
}
grid_rf = GridSearchCV(
    estimator=rf_search,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=5,
    verbose=1,
    n_jobs=-1,
)
grid_rf.fit(X_train, y_train)
print("Best parameters:", grid_rf.best_params_)
print("Best CV macro F1:", grid_rf.best_score_)
